In [63]:
%load_ext autoreload
%autoreload 1
%aimport classes.GaloisField
%aimport classes.GolayDecoder

import numpy as np

from classes.GaloisField import *
from classes.GaloisPoly  import *
from classes.GolayEncoder import GolayEncoder
from classes.GolayDecoder import GolayDecoder

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Generate Galois Field

In [64]:
gf              = GaloisField(1,0b11)
encoder_model   = GolayEncoder()
k               = encoder_model._k
n               = encoder_model._n


Field Closed Succesfully!, 1 Non-Zero Elements


## 1. Codewords Test

### Generate all codewords

In [65]:
encoder_output = None
for w in range(2**k):
    w   = gf.do_unpack(w, bit_width=k)
    cw  = encoder_model.encode(w)
    encoder_output = cw if encoder_output is None else np.vstack((encoder_output, cw))

n_codewords = len(encoder_output)

### Decoding of codewords

In [66]:
n_codewords
#encoder_output 
decoder_model = GolayDecoder()

decoder_output = []

for i in range(n_codewords):
    decoder_output.append(decoder_model.correct(encoder_output[i]))

#print(decoder_output)

Field Closed Succesfully!, 1 Non-Zero Elements


### Error (`o_err`)

In [67]:
o_err           = []
corrected       = []
uncorrectable   = []

o_corrected     = []
o_uncorrectable = []

for i in range(n_codewords):
    # 0: i_rx, 1: o_corrected, 2: o_uncorrectable
    o_err.append(decoder_output[i][0] - encoder_output[i])
    # Flags for uvm
    o_corrected.append(int(decoder_output[i][1]))
    o_uncorrectable.append(int(decoder_output[i][2]))
    # Corrected and uncorrectable bool
    corrected.append(decoder_output[i][1])
    uncorrectable.append(decoder_output[i][2])

# print(o_err, o_corrected, o_uncorrectable)

print(np.array(o_err).shape)
print(np.array(o_corrected).shape)
print(np.array(o_uncorrectable).shape)



(4096, 24)
(4096,)
(4096,)


In [68]:
with open("outputs/top_testing/codewords/codewords_testing_golay_code.svh", "w") as file:

    # Codewords
    file.write("\tconstraint golay_code {\n")
    file.write("\t\trx_data inside {\n")

    for i, cw in enumerate(encoder_output):
        v = gf.do_pack(cw)

        if i == len(encoder_output) - 1:
            file.write(f"\t\t\t24'b{v:024b}\t}};\n")
        else:
            file.write(f"\t\t\t24'b{v:024b}\t,\n")


    # Decoded codewords
    file.write("\n\t\tdecoded_data inside {\n")

    for i, cw in enumerate(decoder_output):
        v = gf.do_pack(cw[0])

        if i == len(decoder_output) - 1:
            file.write(f"\t\t\t24'b{v:024b}\t}};\n")
        else:
            file.write(f"\t\t\t24'b{v:024b}\t,\n")

    # Errors
    file.write("\n\t\terr inside {\n")

    for i, err in enumerate(o_err):
        v = gf.do_pack(err)

        if i == len(o_err) - 1:
            file.write(f"\t\t\t24'b{v:024b}\t}};\n")
        else:
            file.write(f"\t\t\t24'b{v:024b}\t,\n")


    # Corrected received words flag
    file.write("\n\t\tcorrected_data inside {\n")

    for i, corrected in enumerate(o_corrected):
        if i == len(o_corrected) - 1:
            file.write(f"\t\t\t1'b{corrected}\t}};\n")
        else:
            file.write(f"\t\t\t1'b{corrected}\t,\n")

    # Uncorrectable received words flag
    file.write("\n\t\tuncorrectable inside {\n")

    for i, uncorrectable in enumerate(o_uncorrectable):
        if i == len(o_uncorrectable) - 1:
            file.write(f"\t\t\t1'b{uncorrectable}\t}};\n")
        else:
            file.write(f"\t\t\t1'b{uncorrectable}\t,\n")


    file.write("\t};\n")

## 2. Codewords with errors Test

In [69]:
encoder_output 

received_msg_with_error = []

n_errors = 5 # 4 errors limit

for n in range(1, n_errors):
    codewords_n_errors = []
    # Select random n positions
    error_positions = np.random.choice(24, n, replace=False)

    for i in range(n_codewords):
        codeword = encoder_output[i].copy()

        # Put errores
        for pos in error_positions:
            codeword[pos] ^= 1

        # Guardar la codeword completa
        codewords_n_errors.append(codeword)

    received_msg_with_error.append(codewords_n_errors)

#print(received_msg_with_error[0][1][3])

# Dimensions

# received_msg_with_error[0] → codewords with 1 error
# received_msg_with_error[1] → codewords with 2 errors
# received_msg_with_error[2] → codewords with 3 errors
# received_msg_with_error[3] → codewords with 4 errors

# received_msg_with_error[0][0] → first codeword with 1 error
# received_msg_with_error[0][1] → second codeword with 1 error
# codeword with 24-bits

# received_msg_with_error[0][1][0] → first bit of codeword with 1 error

# received_msg_with_error[error_group][codeword][bit]
#                          │            │          │
#                          │            │          └── 0 ... 23
#                          │            │
#                          │            └──────────── 0 ... n_codewords-1
#                          │
#                          └───────────────────────── 0 ... 3


### Decoding of codewords with errors

In [70]:
o_no_cw_err = []
o_no_cw_msg = []
o_no_cw_uncorrectable = []
o_no_cw_corrected = []

for n in range(n_errors - 1):

    # Resultados para esta cantidad de errores
    err_n = []
    msg_n = []
    uncorrectable_n = []
    corrected_n = []

    for i in range(n_codewords):
        received = received_msg_with_error[n][i]
        # Decoder
        decoded = decoder_model.correct(received)
        # Syndrome
        s, q = decoder_model.get_s_q(received)
        # Error pattern
        if decoder_model._gf.do_pack(s) != 0:
            err_n.append(decoder_model.get_error(s, q))
        else:
            err_n.append(0)
        # Message/corrected codeword
        msg_n.append(decoded[0])

        # Flags
        uncorrectable_n.append(int(decoded[1]))
        corrected_n.append(int(decoded[2]))

    # Guardar resultados de este número de errores
    o_no_cw_err.append(err_n)
    o_no_cw_msg.append(msg_n)
    o_no_cw_uncorrectable.append(uncorrectable_n)
    o_no_cw_corrected.append(corrected_n)

### Create files `received words with n-errors`

In [72]:
for n in range(n_errors - 1):

    filename = f"outputs/top_testing/codewords/codewords_testing_golay_{n+1}_error.svh"

    with open(filename, "w") as file:

        # Codewords received with errors
        file.write("\tconstraint golay_code {\n")
        file.write("\t\trx_data inside {\n")

        for i, cw in enumerate(received_msg_with_error[n]):
            v = gf.do_pack(cw)

            if i == len(received_msg_with_error[n]) - 1:
                file.write(f"\t\t\t24'b{v:024b}\t}};\n")
            else:
                file.write(f"\t\t\t24'b{v:024b}\t,\n")


        # Decoded codewords
        file.write("\n\t\tdecoded_data inside {\n")

        for i, cw in enumerate(o_no_cw_msg[n]):
            v = gf.do_pack(cw)

            if i == len(o_no_cw_msg[n]) - 1:
                file.write(f"\t\t\t24'b{v:024b}\t}};\n")
            else:
                file.write(f"\t\t\t24'b{v:024b}\t,\n")


        # Errors
        file.write("\n\t\terr inside {\n")

        for i, err in enumerate(o_no_cw_err[n]):

            # If there is no error pattern, use a 24-bit zero vector
            if err is None:
                err = [0] * 24

            v = gf.do_pack(err)

            if i == len(o_no_cw_err[n]) - 1:
                file.write(f"\t\t\t24'b{v:024b}\t}};\n")
            else:
                file.write(f"\t\t\t24'b{v:024b}\t,\n")


        # Corrected received words flag
        file.write("\n\t\tcorrected_data inside {\n")

        for i, corrected in enumerate(o_no_cw_corrected[n]):
            if i == len(o_no_cw_corrected[n]) - 1:
                file.write(f"\t\t\t1'b{corrected}\t}};\n")
            else:
                file.write(f"\t\t\t1'b{corrected}\t,\n")


        # Uncorrectable received words flag
        file.write("\n\t\tuncorrectable inside {\n")

        for i, uncorrectable in enumerate(o_no_cw_uncorrectable[n]):
            if i == len(o_no_cw_uncorrectable[n]) - 1:
                file.write(f"\t\t\t1'b{uncorrectable}\t}};\n")
            else:
                file.write(f"\t\t\t1'b{uncorrectable}\t,\n")


        file.write("\t};\n")

## golay 24,12 decoder model

In [ ]:
r = encoder_ouput[3576]
# Generate random error for r (word received/transmitted)
r = r ^ np.array([
    [1,0,0,1,0,0,0,1,0,0,0,0]   ,
    [0,0,0,0,0,0,0,0,0,0,0,0]   ]).flatten()

decoder_model = GolayDecoder()

w, corrected, uncorrectable = decoder_model.decode(r)

# r: word with errors
# word decoded (possible codeword),
# flags (corrected, uncorrectable)
# encoder output (word received transmitted)
r, decoder_model.decode(r), encoder_ouput[3576]

NameError: name 'encoder_ouput' is not defined

In [ ]:
# decode all codewords, no errors
for cw in encoder_ouput:
    w, corrected, uncorrectable = decoder_model.decode(cw, full_codeword=True)
    assert(np.all(w == cw))
    assert(not corrected and not uncorrectable)

for cw in encoder_ouput:
    error_seed      = np.random.randint(0, 0b11111)
    # decimal error_seed converted into n-bits
    error           = decoder_model._gf.do_unpack(error_seed, bit_width=decoder_model._n)
    # calculate hamming weight
    error_weight    = decoder_model._gf.hamming_weight(error)
    
    np.random.shuffle(error)
    w, corrected, uncorrectable = decoder_model.decode(cw ^ error, full_codeword=True)
    
    # no errors
    if error_weight == 0:
        assert(np.all(w == cw))
        assert(not corrected and not uncorrectable)
    # 1 to 3 errors
    elif 1 <= error_weight <= 3:
        assert(np.all(w == cw))
        assert(corrected and not uncorrectable)
    # 4 errors
    else:
        assert(not np.all(w == cw))
        assert(not corrected and uncorrectable)

    # five or more errors this decoding fails, recovered bits and flags are invalid


#### Test `golay_err_gen.sv`

Verify the following values:

$$ r_{1} = 0xA5D9A6 $$
$$ r_{2} = 0xA5F9A4 $$
$$ r_{3} = 0xA5C9AA $$

Obtain values corrected.

In [ ]:
s0 = decoder_model._gf.do_pack(decoder_model._G2412B[6])
s1 = decoder_model._gf.do_pack(decoder_model._G2412B[7])

hex(s0 ^ s1)

'0x783'

In [ ]:
import numpy as np

rs = [0xA5D9A6, 0xA5F9A4, 0xA5C9AA]

rs_words = [format(x, '024b') for x in rs]

def str_values_2_binary(r_n):

    # Convert text to bits.
    r_bits = np.array([int(bit) for bit in r_n])

    return r_bits

def array_2_binaryInt(w : np.array):
    # Convert str decode word into hexadecimal
    # 1. numbers into string map(str, w)
    # 2. put together .join()
    # 3. interprete as binary int(str, 2) 2 binary
    w_hex = int(''.join(map(str, w)), 2)
    return w_hex

with open("outputs/decoding/golay_decoding_test_vectors.txt", "w") as f:

    # for within rs_words
    # i: number of iterations
    # r_n: current word of rs_words
    for i, r_n in enumerate(rs_words, start=1):
        r_bits = str_values_2_binary(r_n)

        # Call method .decode
        w, corrected, uncorrectable = decoder_model.decode(
            r_bits,
            full_codeword=True
        )

        w_hex = array_2_binaryInt(w)   # Obtain HEX value from rs array
        r_hex = rs[i-1]

        f.write(f"{r_hex:06X} {w_hex:06X}\n")

        print(
            f"r{i} = 0x{r_hex:06X} | "
            f"corrected_w = 0x{w_hex:06X} | "
            f"corrected = {corrected} | "
            f"uncorrectable = {uncorrectable}"
        )

print("Archivo generado: golay_decoding_test_vectors.txt")

r1 = 0xA5D9A6 | corrected_w = 0xA5C9A5 | corrected = True | uncorrectable = False
r2 = 0xA5F9A4 | corrected_w = 0xA5C9A5 | corrected = True | uncorrectable = False
r3 = 0xA5C9AA | corrected_w = 0xA5C9AA | corrected = False | uncorrectable = True
Archivo generado: golay_decoding_test_vectors.txt


##### Obtain error



In [ ]:
import numpy as np

rs = [0xA5D9A6, 0xA5F9A4, 0xA5C9AA]

rs_words = [format(x, '024b') for x in rs]

with open("outputs/error_gen/golay_err_gen_vector.txt", "w") as f:

    for i, r_n in enumerate(rs_words, start=1):

        r_bits = str_values_2_binary(r_n)
        s, q = decoder_model.get_s_q(r_bits)
        error = decoder_model.get_error(s, q)
        r_hex = rs[i - 1]

        ### Gets
        # sbi, qbi 
        for i, bi in enumerate(decoder_model._G2412B):
            sbi, qbi = decoder_model.get_sbi_qbi(s, q, bi)
            ui    = decoder_model._gf.do_unpack(1<<i, decoder_model._k)[::-1]

        w_syn = decoder_model._gf.hamming_weight(s)
        w_q   = decoder_model._gf.hamming_weight(q)

        _, _, uncorrectable = decoder_model.decode(r_bits)

        # prints
        if error is None:
            print(
                f"r{i} = 0x{r_hex:06X} | "
                f"error_mask = UNCORRECTABLE"
            )
            f.write(f"{r_hex:06X} UNCORRECTABLE\n")
        else:
            hex_error_mask = array_2_binaryInt(error)
            f.write(
                f"{r_hex:06X} "
                f"{hex_error_mask:06X}\n"
            )

            print(
                f"r{i} = 0x{r_hex:06X} | "
                f"error_mask_{i} = "
                f"0x{hex_error_mask:06X}"
            )

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/error_gen/golay_err_gen_vector.txt'

In [ ]:
import numpy as np

rs = [0xA5D9A6, 0xA5F9A4, 0xA5C9AA]
rs_words = [format(x, "024b") for x in rs]

with open("outputs/error_gen/golay_err_gen_vector.txt", "w") as f:

    for idx, r_n in enumerate(rs_words, start=1):

        r_bits = str_values_2_binary(r_n)
        r_hex = rs[idx - 1]

        # Syndrome, q and error
        s, q = decoder_model.get_s_q(r_bits)
        error = decoder_model.get_error(s, q)

        # Find s and q indices independently
        s_idx = None
        q_idx = None
        sbi   = None
        qbi   = None
        s_ui  = None
        q_ui  = None

        #decoder_model._G2412B


## Generation of vectors for testing

In [ ]:
# Matrix
G = decoder_model._G2412B

In [ ]:
decoder_model._gf

# decoder_model._gf.mat_mul(G)